# Otimização Optuna

In [ ]:
from datetime import datetime
import pandas as pd
from src.Classification.Optimizer import ClassificationOptunaOptimizer
from src.Data.Processor import DataStreamProcessor

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES),
]

modelos_para_otimizar = ['LB', 'HAT', 'ARF', 'HT']

def build_stream(dataset_path, selected_features=None):
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=selected_features)

    return processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=[
            'Source IP', 'Source Port', 'Destination IP',
            'Destination Port', 'Protocol', 'Inbound'
        ],
        imputation_method='mediana'
    )


In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')

    for feat_name, selected_feats in configuracoes_features:
        stream, targets, features = build_stream(dataset_path, selected_features=selected_feats)

        for model_name in modelos_para_otimizar:
            optimizer = ClassificationOptunaOptimizer(
                stream=stream,
                n_trials=75,
                target_names=targets,
                n_runs=5
            )

            best_params = optimizer.optimize(
                model_name=model_name,
                warmup_instances=2000,
                experiment_name=nome_experimento,
                num_features=len(features),
                exec_id=id_execucao,
                window_evaluation=100
            )


# Execução Default

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import pandas as pd

from src.Classification.Pipeline import ClassificationExperimentRunner
from src.Classification.Models import get_classification_models
from src.Data.Processor import DataStreamProcessor

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES),
]

def build_stream(dataset_path, selected_features=None):
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=selected_features)

    return processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=[
            'Source IP', 'Source Port', 'Destination IP',
            'Destination Port', 'Protocol', 'Inbound'
        ],
        imputation_method='mediana'
    )


In [ ]:
default_params = {
    'LB': {
        'base_learner': None,
        'ensemble_size': 100,
        'minibatch_size': None,
        'number_of_jobs': None
    },
    'HAT': {
        'grace_period': 200,
        'confidence': 1e-3,
        'tie_threshold': 0.05
    },
    'ARF': {
        'base_learner': None,
        'ensemble_size': 100,
        'max_features': 0.6,
        'lambda_param': 6.0,
        'minibatch_size': None,
        'number_of_jobs': 1,
        'drift_detection_method': None,
        'warning_detection_method': None,
        'disable_weighted_vote': False,
        'disable_drift_detection': False,
        'disable_background_learner': False
    },
    'HT': {
        'grace_period': 200,
        'confidence': 1e-3,
        'tie_threshold': 0.05
    }
}

model_config = {
    'LB': {
        'param_key': 'lb_params',
        'output_name': 'LeveragingBagging'
    },
    'HAT': {
        'param_key': 'hat_params',
        'output_name': 'HoeffdingAdaptiveTree'
    },
    'ARF': {
        'param_key': 'arf_params',
        'output_name': 'AdaptiveRandomForest'
    },
    'HT': {
        'param_key': 'ht_params',
        'output_name': 'HoeffdingTree'
    }
}

modelos_para_executar = ['LB', 'HAT', 'ARF', 'HT']

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')

    for feat_name, selected_feats in configuracoes_features:
        stream, targets, features = build_stream(dataset_path, selected_features=selected_feats)
        schema = stream.get_schema()

        for model_name in modelos_para_executar:
            params = default_params[model_name].copy()
            cfg = model_config[model_name]

            def make_model(run_seed=None, model_name=model_name, params=params, cfg=cfg, schema=schema):
                kwargs = {cfg['param_key']: params}
                models = get_classification_models(
                    schema,
                    selected_models=[model_name],
                    run_seed=run_seed,
                    **kwargs
                )
                return models[cfg['output_name']]

            algoritmos = {
                cfg['output_name']: make_model
            }

            runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)

            runner.run_classification_evaluation(
                stream=stream,
                algorithms=algoritmos,
                window_evaluation=100,
                warmup_instances=2000,
                title=nome_experimento,
                algorithm_params=params,
                is_optimized=False,
                num_features=len(features),
                exec_id=id_execucao
            )


## Hoeffding Tree (HT)

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Results.Plots import Plots

# Especifique os caminhos dos 4 arquivos CSV do algoritmo que deseja resumir
csvs_aif = {
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Otimizado_FullFeatures.csv": "20260416_1737",
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Default_FullFeatures.csv": "20260416_1901",
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Default_33Features.csv": "20260416_1902",
}

# Inicializamos a classe Plots
plots = Plots(target_names=['Normal', 'Ataque'])

plots.plot_summary_table(
    csv_dict=csvs_aif, 
    algo_name="AdaptiveIsolationForest",
    exclude_blocks=['25'],
    include_blocks=['200', '1000']
)

## Adaptive Random Forest Classifier (ARF)

## Leveraging Bagging (LB)

## Hoeffding Adaptive Tree (HAT)